## Q1

In [2]:
import pandas as pd
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt

state = pd.read_csv("state.csv")
y = state.loc[:, "Life.Exp"].to_numpy()
X = state.drop(columns=[state.columns[0], "Life.Exp"]).to_numpy()

X_std = (X - X.mean(axis=0)) / X.std(axis=0)
p = X.shape[1]

In [3]:
with pm.Model() as q1_model:
    beta = pm.Normal("beta", 0, tau=0.1, shape=p)

    sigma = pm.Exponential("sigma", 0.01)
    intercept = pm.Normal("intercept", 0, tau=0.0001)

    mu = intercept + pm.math.dot(X, beta)

    pm.Normal("lik", mu, sigma, observed=y)

    trace = pm.sample(10000, tune=1000, target_accept=0.9)

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [beta, sigma, intercept]


Output()

Sampling 4 chains for 1_000 tune and 10_000 draw iterations (4_000 + 40_000 draws total) took 21 seconds.


In [4]:
az.summary(trace, hdi_prob=0.95, round_to=6)

,mean,sd,hdi_2.5%,hdi_97.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
beta[0],0.000052,0.000030,-0.000008,0.000111,0.000000,0.000000,24534.352397,26399.825459,1.000069
beta[1],-0.000021,0.000254,-0.000521,0.000475,0.000002,0.000001,22535.860039,24274.393223,1.000304
beta[2],0.037102,0.379157,-0.720323,0.773465,0.003050,0.002112,15446.873828,21757.811857,1.000039
beta[3],-0.300691,0.048347,-0.394758,-0.204512,0.000314,0.000252,23785.483840,25055.971647,1.000070
beta[4],0.049300,0.024167,0.000255,0.094809,0.000189,0.000133,16397.590585,22524.153713,1.000200
beta[5],-0.005720,0.003234,-0.012217,0.000504,0.000024,0.000017,18812.153929,23964.812265,1.000161
beta[6],-0.000000,0.000002,-0.000003,0.000003,0.000000,0.000000,18608.237712,22982.983536,1.000101
intercept,70.914699,1.805879,67.425617,74.524514,0.015937,0.010991,12841.763292,17485.380443,1.000144
sigma,0.767868,0.086538,0.607729,0.940695,0.000550,0.000490,25397.783729,26391.731627,1.000187


In [5]:
with pm.Model() as q1_model_std:
    beta = pm.Normal("beta", 0, tau=0.1, shape=p)

    sigma = pm.Exponential("sigma", 0.05)
    intercept = pm.Normal("intercept", 0, tau=0.0001)

    mu = intercept + pm.math.dot(X_std, beta)

    pm.Normal("lik", mu, sigma, observed=y)

    trace_std = pm.sample(10000, tune=1000)

az.summary(trace_std, hdi_prob=0.95, round_to=7)

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [beta, sigma, intercept]


Output()

Sampling 4 chains for 1_000 tune and 10_000 draw iterations (4_000 + 40_000 draws total) took 4 seconds.


,mean,sd,hdi_2.5%,hdi_97.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
beta[0],0.227726,0.132110,-0.027765,0.490337,0.000736,0.000653,32264.483154,29711.663295,1.000086
beta[1],-0.013827,0.153270,-0.316040,0.291671,0.000849,0.000757,32601.560406,28925.378656,1.000097
beta[2],0.017835,0.227337,-0.443435,0.453416,0.001437,0.001137,25033.174451,27077.785920,0.999963
beta[3],-1.097278,0.175914,-1.444526,-0.752877,0.000978,0.000931,32433.768089,27437.188287,1.000075
beta[4],0.390332,0.192334,0.016195,0.769791,0.001171,0.000944,27028.415546,27742.144133,1.000088
beta[5],-0.294910,0.167031,-0.621349,0.035177,0.000987,0.000808,28656.829770,28909.333274,1.000014
beta[6],-0.005774,0.145428,-0.287489,0.287326,0.000878,0.000705,27512.617829,28416.612369,1.000176
intercept,70.878334,0.109754,70.667772,71.099444,0.000532,0.000583,42722.412287,28073.510293,1.000069
sigma,0.767043,0.087217,0.608571,0.943620,0.000506,0.000466,30356.579196,28310.655722,1.000022


In [6]:
with pm.Model() as q1_ssvs:
    delta = pm.Bernoulli("delta", p=0.5, shape=p)
    alpha = pm.Normal("alpha", 0, tau=0.1, shape=p)
    beta = pm.Deterministic("beta", delta * alpha)
    intercept = pm.Normal("intercept", 0, tau=0.0001)

    sigma = pm.Exponential("sigma", 0.01)

    mu = intercept + pm.math.dot(X_std, beta)

    pm.Normal("lik", mu, sigma, observed=y)

    trace_ssvs = pm.sample(10000, target_accept=0.99, tune=1000)

az.summary(trace_ssvs, var_names="delta", hdi_prob=0.95, kind="stats")

Multiprocess sampling (4 chains in 4 jobs)
CompoundStep
>BinaryGibbsMetropolis: [delta]
>NUTS: [alpha, intercept, sigma]


Output()

Sampling 4 chains for 1_000 tune and 10_000 draw iterations (4_000 + 40_000 draws total) took 62 seconds.
/Users/aaron/miniforge3/envs/pymc_spr26/lib/python3.14/site-packages/arviz/stats/diagnostics.py:596: RuntimeWarning: invalid value encountered in scalar divide
  (between_chain_variance / within_chain_variance + num_samples - 1) / (num_samples)


,mean,sd,hdi_2.5%,hdi_97.5%
delta[0],0.292,0.455,0.0,1.0
delta[1],0.088,0.283,0.0,1.0
delta[2],0.089,0.285,0.0,1.0
delta[3],1.000,0.000,1.0,1.0
delta[4],0.633,0.482,0.0,1.0
delta[5],0.466,0.499,0.0,1.0
delta[6],0.052,0.222,0.0,1.0


In [7]:
chains = 4
draws = 10000
rows = chains * draws

deltas = trace_ssvs.posterior.delta.to_numpy()
models, counts = np.unique(
    deltas.reshape((rows, p)), axis=0, return_counts=True
)

results = zip(models, counts)

for model, count in sorted(results, key=lambda x: x[1], reverse=True):
    print(f"{model}: prob={count/rows:.3}")

[0 0 0 1 1 1 0]: prob=0.238
[0 0 0 1 1 0 0]: prob=0.151
[0 0 0 1 0 0 0]: prob=0.116
[1 0 0 1 1 0 0]: prob=0.081
[1 0 0 1 0 0 0]: prob=0.0777
[1 0 0 1 1 1 0]: prob=0.0608
[0 0 0 1 0 1 0]: prob=0.0442
[0 1 0 1 0 0 0]: prob=0.0222
[0 1 0 1 0 1 0]: prob=0.021
[0 0 1 1 1 1 0]: prob=0.0174
[1 0 0 1 0 1 0]: prob=0.0167
[0 0 1 1 0 1 0]: prob=0.0164
[0 0 1 1 1 0 0]: prob=0.0131
[1 0 1 1 1 0 0]: prob=0.0124
[0 1 0 1 1 1 0]: prob=0.0118
[0 0 0 1 1 1 1]: prob=0.0107
[0 0 0 1 1 0 1]: prob=0.00745
[0 0 1 1 0 0 0]: prob=0.00743
[0 1 0 1 1 0 0]: prob=0.00728
[1 1 0 1 0 0 0]: prob=0.00725
[0 0 0 1 0 0 1]: prob=0.00662
[1 0 1 1 0 0 0]: prob=0.00502
[0 0 0 1 0 1 1]: prob=0.00495
[1 0 0 1 0 0 1]: prob=0.00465
[1 0 0 1 1 0 1]: prob=0.0039
[1 0 1 1 1 1 0]: prob=0.0036
[1 1 0 1 1 0 0]: prob=0.00335
[1 0 1 1 0 1 0]: prob=0.00258
[1 0 0 1 0 1 1]: prob=0.0025
[1 1 0 1 1 1 0]: prob=0.0024
[1 0 0 1 1 1 1]: prob=0.00235
[1 1 0 1 0 1 0]: prob=0.00235
[0 1 1 1 0 1 0]: prob=0.00217
[0 0 1 1 0 1 1]: prob=0.0016
[0 1 1